# Analyzing Entropy Differences Based on Various Group and Discursive Factors

In [ ]:
import os
import re

HATE_TARGET = 'Islamophobia'
DATA_PATH = '../../data_aggregate/'
RESIDUAL_PATH = 'residual-parents.pt'
REPORT_NAME = 'report-parents.csv'
target_column = '_about_Muslim_people'
lollipop_vis_name = 'IHS-expected-effects-parents.png'
REPORTING_PATH = '../reporting/{}'.format(HATE_TARGET) #os.path.join('../data/reports/', HATE_TARGET.lower())

if not os.path.exists(REPORTING_PATH):
    os.mkdir(REPORTING_PATH)

## Main Analyses and Results

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import statsmodels.formula.api as smf
from datetime import datetime as dt

df = pd.read_csv(os.path.join(DATA_PATH, HATE_TARGET+'-cleaned.csv'))
df = df.loc[
    (df['nx'] >= 5)
    & (df['ny'] >= 5)
    # & (df['comment_delta_abs'] <= 20)
] # limit by comment size

df['is_parent'] = df['is_parent'].astype(bool)
# df['is_sibling'].loc[df['is_parent']] = False
df['other_thread'] = (~df['is_parent']) & (~df['is_sibling'])

df['is_parent_'] = df['is_parent'].astype(int)
df['is_sibling_'] = df['is_sibling'].astype(int)
df['other_thread_'] = df['other_thread'].astype(int)

In [ ]:
df = df.loc[
    df['is_parent'] #| 
    # df['is_sibling']
    & df['x_subreddit'].isin(['2westerneurope4u', '4chan', 'fingmemes'])
]

In [ ]:
# df.isna().sum()
df.shape[0] ,df['x'].nunique(), df['y'].nunique() 

In [ ]:
df.head()

In [ ]:
df['x_target'] = df['x'+target_column]
df['y_target'] = df['y'+target_column]

In [ ]:
sub_names = df['x_subreddit'].unique()
convert_sub_names = {x:i for i,x in enumerate(np.random.choice(sub_names, size=(len(sub_names),), replace=False))}
df['x_subreddit'] = [convert_sub_names[x] for x in tqdm(df['x_subreddit'].values)]

In [ ]:
df['after_october_7'] = (df['x_comment_created_at'] >= 1696716000).astype(int)

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.head()

## Removing bad IDs

This includes any instances in which the text consists entirely of either `<br>[removed]`, `<br>[deleted]` or `<br>[deleted by moderator]`

In [ ]:
import json

In [ ]:
# get text file
TEXT_FILE = os.path.join(
    DATA_PATH,
    HATE_TARGET.lower() + '-texts.tsv'
)
dft = pd.read_table(TEXT_FILE, sep='\t')

# relabel x-column
f = open(
    os.path.join(
        DATA_PATH,
        'x-'+HATE_TARGET+'.json'
    ),
    'r'
)
rev_conversion_dic = json.loads(f.read())
conversion_dic = {v:k for k,v in rev_conversion_dic.items()}
f.close()

df['x'] = [conversion_dic[xid] for xid in tqdm(df['x'].values)]

In [ ]:
sel = dft['body'].apply(lambda x: ('[removed]' in str(x)) or ('[deleted]' in str(x)) or ('by moderator]' in str(x)))
sel = dft['comment_id'].loc[sel].unique()
print(df.shape)
df = df.loc[(~df['x'].isin(sel)) & (~df['y'].isin(sel))]
df.shape

In [ ]:
del dft

### AHS Frequency

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.histplot(
    data=df.loc[df['x'+target_column].astype(bool)].drop_duplicates(subset=['x']),
    x='x_probs'
)
plt.show()

In [ ]:
df_ = df.loc[df['x'+target_column].astype(bool)].drop_duplicates(subset=['x']).copy()
df_['hate_level'] = 'Not HS'
df_['hate_level'].loc[df_['x_probs'] >= .25] = 'Not likely HS'
df_['hate_level'].loc[df_['x_probs'] >= .5] = 'Possibly HS'
df_['hate_level'].loc[df_['x_probs'] >= .75] = 'Quite Possibly HS'
df_['hate_level'].loc[df_['x_probs'] >= .9] = 'HS'
df_['hate_level'].value_counts()

### Main LME Analysis

In [ ]:
############### Categorical Division of Influence on Comment Delta effects
############### Categorical Division of Influence on Comment Delta effects
# current paper version
# model = "H ~ x_comment_ups + y_comment_ups + after_october_7*(x_probs*x_target) + after_october_7*(y_probs*y_target) + comment_delta + (comment_delta|is_parent_) + (comment_delta|is_sibling_) + nx + ny + (1|x_subreddit) + (1|x_user) + (1|y_user) + (2|x_submission_id)"
model = "H ~ x_comment_ups + after_october_7*(x_probs*x_target) + comment_delta + nx + ny + (1|x_subreddit) + (1|x_user) + (1|y_user)"

###############

start = dt.now()
md = smf.mixedlm(model, data=df, groups=df['x'])
mdf = md.fit()
print('completed in:', dt.now()-start)

Reporting on the model outputs in a dataframe

In [ ]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

In [ ]:
reporting.to_csv(os.path.join(REPORTING_PATH,REPORT_NAME), encoding='utf-8')

reporting['Var'] = reporting.index.values
with open(os.path.join(REPORTING_PATH, REPORT_NAME.replace('.csv', '.txt')), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].loc[:reporting.index[-2]].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
    f.write(txt)
    f.close()

In [ ]:
printable_reporting = reporting.copy()
printable_reporting.index = [
    i.replace('_', ' ').replace('probs', 'HS rating').replace('target', target_column[1:].replace('_', ' ')).replace('delta', '$\\Delta$').replace('comment ups', 'likes received').replace('x subreddit', 'subreddit').replace('Group Var', '1 | x')
    for i in printable_reporting.index
]
printable_reporting.to_csv(os.path.join(REPORTING_PATH,REPORT_NAME.replace('.csv', '-for_paper.csv')), encoding='utf-8')

### Testing model significance

#### Targeted hate speech

In [ ]:
# pre october 7th
test_matrix = np.zeros(shape=(len(mdf.params)))
sel = np.array([
    (('x_target' in k) or ('x_probs' in k)) and ('after_october_7' not in k) 
    for k in mdf.params.keys()
])
test_matrix[sel] = 1

res = mdf.f_test(test_matrix)
print(res.df_denom)
res

In [ ]:
# after october 7th
test_matrix = np.zeros(shape=(len(mdf.params)))
sel = np.array([
    (('x_target' in k) or ('x_probs' in k)) or ('after_october_7' == k)
    for k in mdf.params.keys()
])
test_matrix[sel] = 1

res = mdf.f_test(test_matrix)
print(res.df_denom)
res

## Additional Visualizations/Analyses

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kruskal

### Lolipop effects plot

In [ ]:
import pandas as pd
import numpy as np

# reporting = pd.read_csv('data/reports/antisemitism/report.csv')
reporting = pd.read_csv(os.path.join(REPORTING_PATH,REPORT_NAME))
reporting.index = reporting['Unnamed: 0'].values

In [ ]:
import plotly.graph_objs as go

marker_offset = 0.0004

def offset_signal(signal, marker_offset):
    if abs(signal) <= marker_offset:
        return 0
    return signal - marker_offset if signal > 0 else signal + marker_offset

def plotly_lollipop(df, label_col, length_col, save_path=None, plot_title='', color='blue', marker_size=2):
    points = df[length_col].to_list()
    heights = list(range(len(df)))
    
    data = [
    go.Scatter(
            x=points,
            y=heights,
            mode='markers',
            marker=dict(
                color=color,
                size=marker_size
            )
        )
    ]

    layout = go.Layout(
    shapes=[dict(
            type='line',
            xref='x',
            yref='y',
            y0=i,
            x0=0,
            y1=i,
            x1=offset_signal(points[i], marker_offset),
            line=dict(
                color=color,
                width=1.5
            )
        ) for i in range(len(points))],
    )

    fig = go.Figure(data, layout)

    for idx in range(len(fig.data)):
        fig.data[idx].y = df_param['cond'].to_list()

    fig.add_vline(x=0, line_width=3, line_color="maroon")
    
    return fig

In [ ]:
def lollipop_chart(df, label_col, length_col, save_path=None, aspect=1/15, plot_title='Predicted change in H'):
    sns.set_style('darkgrid')
    plt.hlines(y=df.index, xmin=0, xmax=df[length_col].values)
    plt.plot(df[length_col].values, df.index, 'o')
    plt.yticks(df.index, df[label_col].values, rotation=.45, fontsize='small')
    plt.axvline(color='maroon')

    xlim_delta = df[length_col].__abs__().max() + .1
    plt.xlim(-xlim_delta, xlim_delta)
    plt.gca().set_aspect(aspect)
    plt.tight_layout()
    plt.xlabel(plot_title)
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
df_param = [
    # x HS
    ['HS', (reporting['coefs'].loc[['x_probs']]).sum()],
    
    # x IHS
    ['IHS', (
        reporting['coefs'].loc[['x_probs', 'x_target','x_probs:x_target']] #* (reporting['p'].loc[['x_probs', 'x_target','x_probs:x_target']] < .01)
    ).sum()],
    
    # x IHS post-october 7th
    ['IHS after Oct. 7, 2023', (
        reporting['coefs'].loc[['x_probs', 'x_target', 'x_probs:x_target','after_october_7', 'after_october_7:x_target', 'after_october_7:x_probs', 'after_october_7:x_probs:x_target']] #* (reporting['p'].loc[['x_probs', 'x_target', 'x_probs:x_target','after_october_7', 'after_october_7:x_target', 'after_october_7:x_probs', 'after_october_7:x_probs:x_target']] < .01)
    ).sum()],
    
    # # Y HS
    # ['Y HS', (reporting['coefs'].loc[['y_probs']]).sum()],
    # 
    # # Y IHS
    # ['Y IHS', (
    #     reporting['coefs'].loc[[ 'y_probs', 'y_target', 'y_probs:y_target']] #* (reporting['p'].loc[[ 'y_probs', 'y_target', 'y_probs:y_target']] < .01)
    # ).sum()],
    # 
    # # Y IHS post-october 7th
    # ['Y IHS after Oct. 7, 2023', (
    #     reporting['coefs'].loc[['y_probs', 'y_target', 'y_probs:y_target','after_october_7', 'after_october_7:y_target', 'after_october_7:y_probs','after_october_7:y_probs:y_target',]] #* (reporting['p'].loc[['y_probs', 'y_target', 'y_probs:y_target','after_october_7', 'after_october_7:y_target', 'after_october_7:y_probs','after_october_7:y_probs:y_target',]] <.01)
    # ).sum()], 
][::-1]

df_param = pd.DataFrame(
    np.array(df_param, dtype=object),
    columns=['cond', '$Delta$ H']
)

In [ ]:
# lollipop_chart(
#     df=df_param,
#     label_col='cond',
#     length_col='$Delta$ H',
#     save_path=lollipop_vis_name,
#     aspect=1/1.15,
#     plot_title=''
# )

In [ ]:
fig = plotly_lollipop(
    df=df_param,
    label_col='cond',
    length_col='$Delta$ H',
    marker_size=10
)

fig.show()

In [ ]:
fig.write_html('ihs-parent.html')